### Read experimental result data

In [1]:
# read parquet file
import polars as pl
# df = pl.read_parquet("../../experiments/16_10_result.parquet")
df = pl.read_parquet("/Users/shinomilab-mac-pro/seokyeong/diffusion_seokyeong/test/twitch_ENGB.parquet")
print(df)

# read metadata of the parquet file
import pyarrow.parquet as pq
# meta = pq.read_metadata("../test/result.parquet")
meta = pq.read_metadata("/Users/shinomilab-mac-pro/seokyeong/diffusion_seokyeong/test/twitch_ENGB.parquet")

# You can check versions depending on libraries 
print(meta.metadata)

shape: (524_288, 5)
┌─────┬──────┬───────┬───────┬───────────────┐
│ id  ┆ ip   ┆ us_id ┆ msg   ┆ mean_num_xact │
│ --- ┆ ---  ┆ ---   ┆ ---   ┆ ---           │
│ u64 ┆ u64  ┆ u64   ┆ str   ┆ f64           │
╞═════╪══════╪═══════╪═══════╪═══════════════╡
│ 0   ┆ 6623 ┆ 0     ┆ 30000 ┆ 2.06          │
│ 0   ┆ 6623 ┆ 0     ┆ 31000 ┆ 2.34          │
│ 0   ┆ 6623 ┆ 0     ┆ 11000 ┆ 2.23          │
│ 0   ┆ 6623 ┆ 0     ┆ 01000 ┆ 2.37          │
│ 0   ┆ 6623 ┆ 0     ┆ 21000 ┆ 2.52          │
│ …   ┆ …    ┆ …     ┆ …     ┆ …             │
│ 511 ┆ 2821 ┆ 31    ┆ 32333 ┆ 1.66          │
│ 511 ┆ 2821 ┆ 31    ┆ 03333 ┆ 1.54          │
│ 511 ┆ 2821 ┆ 31    ┆ 13333 ┆ 1.65          │
│ 511 ┆ 2821 ┆ 31    ┆ 23333 ┆ 1.51          │
│ 511 ┆ 2821 ┆ 31    ┆ 33333 ┆ 1.43          │
└─────┴──────┴───────┴───────┴───────────────┘
{b'version_runner': b'0.1.1', b'version_core': b'0.1.1', b'version': b'0.1.0', b'ARROW:schema': b'/////0wBAAAQAAAAAAAKAAwACgAJAAQACgAAABAAAAAAAQQACAAIAAAABAAIAAAABAAAAAUAAADsAAAAsAA

### Compute solutions and/or values of optimal, RAM, and URS policies

In [2]:
EV_BY_OPT = "ev_by_opt"
EV_BY_RAM = "ev_by_ram"
EV_BY_URS = "ev_by_urs"
OPT_MEAN = "mean_of_opt_values"
RAMEV_OPTEV_RATIO = "ram ev / opt ev"
URSEV_OPTEV_RATIO = "urs ev / opt ev"
RAMEV_OPTM_RATIO = "ram ev / opt mean"
URSEV_OPTM_RATIO = "urs ev / opt mean"

MEAN_NUM_XACT = "mean_num_xact"

lf_opt = (
    df.lazy()
    .filter(pl.col(MEAN_NUM_XACT) == pl.col(MEAN_NUM_XACT).max().over(["us_id", "ip"]))
    .unique(["ip", "us_id", MEAN_NUM_XACT])
    .select([pl.col("us_id"), pl.col("ip"), pl.col("msg").name.prefix("argmax_"), pl.col(MEAN_NUM_XACT).name.prefix("max_")])
)
lf_opt_mod = (df.lazy()
    .join(lf_opt, left_on=pl.col("msg"), right_on=pl.col("argmax_msg"))
    .group_by("ip_right", "us_id_right")
    .agg(pl.col(MEAN_NUM_XACT).mean())
    .join(lf_opt, left_on=["ip_right", "us_id_right"], right_on=["ip", "us_id"])
    .select([MEAN_NUM_XACT, "max_" + MEAN_NUM_XACT])
    .mean()
    .select([pl.col(MEAN_NUM_XACT).alias(EV_BY_OPT), pl.col("max_" + MEAN_NUM_XACT).alias(OPT_MEAN)])
)
lf_rob = (
    df.lazy()
    .group_by("msg")
    .agg(pl.col(MEAN_NUM_XACT).mean())
    .filter(pl.col(MEAN_NUM_XACT) == pl.col(MEAN_NUM_XACT).max())
    .select(pl.col(MEAN_NUM_XACT).alias(EV_BY_RAM))
)
lf_urs = (
    df.lazy()
    .select(MEAN_NUM_XACT)
    .mean()
    .select(pl.col(MEAN_NUM_XACT).alias(EV_BY_URS))
)
eval_df = (
    pl.concat([lf_opt_mod, lf_rob, lf_urs], how="horizontal")
    .with_columns([
        (pl.col(EV_BY_RAM) / pl.col(EV_BY_OPT)).alias(RAMEV_OPTEV_RATIO),
        (pl.col(EV_BY_URS) / pl.col(EV_BY_OPT)).alias(URSEV_OPTEV_RATIO),
        (pl.col(EV_BY_RAM) / pl.col(OPT_MEAN)).alias(RAMEV_OPTM_RATIO),
        (pl.col(EV_BY_URS) / pl.col(OPT_MEAN)).alias(URSEV_OPTM_RATIO),
    ])
).collect().transpose(include_header=True).rename({"column": "key", "column_0": "value"})

In [3]:
lf_opt_mod = (df.lazy()
    .join(lf_opt, left_on=pl.col("msg"), right_on=pl.col("argmax_msg"))#普通のdfにlf_optで事後確認した最適なmsgを他にip,us_idに適用されたのがあればそれをあわせて平均化。
    .group_by("ip_right", "us_id_right")#他のip,user_stateにもoptのmsgが通用するかという問い
    .agg(pl.col(MEAN_NUM_XACT).mean())
    .join(lf_opt, left_on=["ip_right", "us_id_right"], right_on=["ip", "us_id"])#ipとus_idにピッタリなmsg
    .mean()
    .select([pl.col(MEAN_NUM_XACT).alias("ip,user_state関係なく、最適なmsgをランダムに適用したxact期待値"),
             pl.col("max_" + MEAN_NUM_XACT).alias("ipとuser_stateに最適なmsg適用後の期待値")])
)
lf_opt_mod.mean().collect()

"ip,user_state関係なく、最適なmsgをランダムに適用したxact期待値",ipとuser_stateに最適なmsg適用後の期待値
f64,f64
3.923933,5.808379


ipだけぴったりのmsgだけ適用すると?

In [4]:
ip_opt = (
    df.lazy()
    .filter(pl.col(MEAN_NUM_XACT) == pl.col(MEAN_NUM_XACT).max().over(["ip"]))
    .unique(["ip", MEAN_NUM_XACT])
    .select([pl.col("ip"), pl.col("msg"), pl.col(MEAN_NUM_XACT)])
).collect()


In [5]:
ip_opt

ip,msg,mean_num_xact
u64,str,f64
6623,"""01333""",10.79
4752,"""00333""",19.17
5631,"""01033""",14.98
5603,"""11233""",48.02
3905,"""22333""",26.49
…,…,…
3543,"""00333""",13.94
5785,"""00023""",27.07
2821,"""00233""",4.16


In [6]:
ip_opt.select(pl.col("mean_num_xact").mean().alias("ip_opt_msg_xact"))#平均値がipとuser_stateより高い

ip_opt_msg_xact
f64
14.145


In [7]:
ip_opt.group_by("msg").len()

msg,len
str,u32
"""00033""",1
"""11233""",1
"""22333""",1
"""23333""",2
"""00333""",2
…,…
"""02332""",1
"""22332""",1
"""00023""",1


In [8]:
ip_opt_list=(ip_opt.unique("msg").select(pl.col("msg")))["msg"].to_list()

In [9]:
ip_opt_list

['11233',
 '00023',
 '01333',
 '00033',
 '22332',
 '00333',
 '22333',
 '01033',
 '00233',
 '02332',
 '00000',
 '23333',
 '01023']

ipだけぴったりのmsgだけ適用すると外部行動者数が増加した

In [10]:
ip_opt_mod = (df.lazy()#ipにピッタリなmsgを他のシナリオに適用した場合。
    .join(ip_opt.lazy(), left_on=pl.col("msg"), right_on=pl.col("msg"))
    .group_by("ip_right").agg(pl.col("mean_num_xact").mean()).sort("mean_num_xact",descending=True)
    .select(pl.col("ip_right").alias("ip"),pl.col("mean_num_xact").alias("他のシナリオにmsgを適用したxact"))).collect()
ip_opt_frame=ip_opt_mod.join(ip_opt,left_on="ip",right_on="ip").select(pl.col("ip"),pl.col("msg"),pl.col("他のシナリオにmsgを適用したxact")).sort("他のシナリオにmsgを適用したxact",descending = True)
ip_opt_frame

ip,msg,他のシナリオにmsgを適用したxact
u64,str,f64
2872,"""00033""",4.755977
5785,"""00023""",4.630801
5631,"""01033""",4.500859
2821,"""00233""",4.454023
4236,"""00233""",4.454023
…,…,…
775,"""02332""",3.731758
1035,"""23333""",3.68291
5551,"""23333""",3.68291


In [11]:
ip_opt_frame.select(pl.col("他のシナリオにmsgを適用したxact").mean().alias("他のシナリオにmsgを適用したxactの期待値"))

他のシナリオにmsgを適用したxactの期待値
f64
4.022264


In [12]:
eval_df

key,value
str,f64
"""ev_by_opt""",3.923933
"""mean_of_opt_values""",5.808379
"""ev_by_ram""",4.755977
"""ev_by_urs""",3.277229
"""ram ev / opt ev""",1.212043
"""urs ev / opt ev""",0.83519
"""ram ev / opt mean""",0.818813
"""urs ev / opt mean""",0.564224


In [15]:
###ram
lf_rob_details = (
    df.lazy()
    .group_by("msg")
    .agg(pl.col(MEAN_NUM_XACT).mean())
    .sort(by="mean_num_xact", descending=True)
    # 上位5行を取得
).collect()
lf_rob_details
# lf_rob_details.filter(pl.col("mean_num_xact")>=pl.col("mean_num_xact").mean()*1.8)

msg,mean_num_xact
str,f64
"""00033""",4.755977
"""00023""",4.630801
"""00133""",4.565859
"""10033""",4.553535
"""00123""",4.52707
…,…
"""22000""",1.909512
"""21000""",1.904863
"""20000""",1.894277


In [16]:
lf_rob_details.select(pl.col("mean_num_xact").mean())

mean_num_xact
f64
3.277229


In [17]:
# read parquet file
import polars as pl
# user_analysis= pl.read_parquet("../../experiments/user_analysis.parquet")
user_analysis= pl.scan_parquet("/Users/shinomilab-mac-pro/seokyeong/diffusion_seokyeong/test/twitch_DE_user_analysis.parquet")
user_analysis


In [26]:
ram_msg = lf_rob_details.filter(pl.col("mean_num_xact")>=4.4).select(pl.col("msg"))
ram_msg_list =ram_msg["msg"].to_list()

ram_msg_us_an = user_analysis.lazy().filter(pl.col("msg").is_in(ram_msg_list)).collect()

# # reverse_msg_list = lf_rob_details.lazy().sort("mean_num_xact",descending=True).select(pl.col("msg"))[-10:].to_list()
# # reverse_msg_us_an = user_analysis.lazy().filter(pl.col("msg").is_in(reverse_msg_list)).collect()


In [27]:
ram_msg_list

['00033',
 '00023',
 '00133',
 '10033',
 '00123',
 '10023',
 '01033',
 '20033',
 '00223',
 '00233',
 '01133',
 '10123',
 '01023',
 '10133']

In [28]:
lf_opt.sort("max_mean_num_xact",descending=True)

In [34]:
lf_opt_list=(lf_opt
             .sort("max_mean_num_xact",descending=True).filter(pl.col("max_mean_num_xact")>3.277229

*1.8).collect())

lf_opt_list.group_by("argmax_msg").len().sort("len",descending=True)["argmax_msg"].to_list()
# lf_opt.sort("max_mean_num_xact",descending=True).collect()[:25]

['00033',
 '33333',
 '10033',
 '00023',
 '00133',
 '23333',
 '01033',
 '00223',
 '00003',
 '02233',
 '20023',
 '00233',
 '20133',
 '32333',
 '11133',
 '10003',
 '10023',
 '00333',
 '01023',
 '11223',
 '00123',
 '10013',
 '21233',
 '12233',
 '30023',
 '01233',
 '20033',
 '10333',
 '01133',
 '20323',
 '00013',
 '10223',
 '00022',
 '21333',
 '30033',
 '20003',
 '01123',
 '11023',
 '20123',
 '21123',
 '02013',
 '30102',
 '32323',
 '10123',
 '11033',
 '22332',
 '22133',
 '01333',
 '02333',
 '00132',
 '20333',
 '11333',
 '30213',
 '11233',
 '31233',
 '33223',
 '22003',
 '02303',
 '23013',
 '00113',
 '01213',
 '31323',
 '01113',
 '03023',
 '22122',
 '13033',
 '20122',
 '33233',
 '00332',
 '01003',
 '11323',
 '01223',
 '21023',
 '10032',
 '31133',
 '02133',
 '32223',
 '02033',
 '31333',
 '30013',
 '21133',
 '23033',
 '13333',
 '20102',
 '21033',
 '33123',
 '33033',
 '22333']

In [35]:
ip_opt_list

['11233',
 '00023',
 '01333',
 '00033',
 '22332',
 '00333',
 '22333',
 '01033',
 '00233',
 '02332',
 '00000',
 '23333',
 '01023']

00003,00033,00013,00002,00012は共通
つまり、少なくとも3ラウンドまで内部行動の一番強のメッセージを連続に伝播して、徐々に外部行動強のメッセージを伝播するのが外部行動者数を最大化してくれる。

In [18]:
reverse_msg_list

['33230',
 '23331',
 '32330',
 '23320',
 '33333',
 '33332',
 '33331',
 '33320',
 '23330',
 '33330']

In [36]:
ratio_xact_share_per = (user_analysis.lazy()
.select((pl.col("num_xact_of_users").mean() / pl.col("num_share_of_users").mean()).alias("xact_share_ratio")).collect())
xact_per_mean = (user_analysis.lazy().select(pl.col("num_xact_of_users").mean().alias("mean_num_xact_of_users")).collect())
share_per_mean = (user_analysis.lazy().select(pl.col("num_share_of_users").mean().alias("mean_num_share_of_users")).collect())

In [37]:
concat = pl.concat([xact_per_mean,share_per_mean,ratio_xact_share_per],how="horizontal")
concat

mean_num_xact_of_users,mean_num_share_of_users,xact_share_ratio
f64,f64,f64
0.10943,0.151174,0.723865


In [38]:
ram_ratio = (user_analysis.lazy().filter(pl.col("msg").is_in(ram_msg_list))
        .select((pl.col("num_xact_of_users").mean() / pl.col("num_share_of_users").mean()).alias("xact_share_ratio")))
ram_mean_xact_of_users=user_analysis.lazy().filter(pl.col("msg").is_in(ram_msg_list)).select(pl.col("num_xact_of_users").mean().alias("mean_num_xact_of_users"))
ram_mean_share_of_users=user_analysis.lazy().filter(pl.col("msg").is_in(ram_msg_list)).select(pl.col("num_share_of_users").mean().alias("mean_num_share_of_users"))
ram = (
    pl.concat([ram_mean_xact_of_users, ram_mean_share_of_users, ram_ratio], how="horizontal").collect())
ram


mean_num_xact_of_users,mean_num_share_of_users,xact_share_ratio
f64,f64,f64
0.149638,0.110904,1.349257


In [39]:
xact_share=(user_analysis.lazy()
.group_by(["ip","us_id","msg"]).agg(pl.col("num_xact_of_users").sum().alias("xact"),pl.col("num_share_of_users").sum().alias("share"))
.group_by("msg").agg(pl.col("xact").mean(),pl.col("share").mean()).sort("xact",descending = True)
).collect()


In [40]:
xact_share_mean= xact_share.select(pl.col("xact").mean(),pl.col("share").mean())
xact_share_ratio=xact_share.select([(pl.col("xact").mean()/pl.col("share").mean()).alias("xact/share")])
combine1 = pl.concat([xact_share_mean,xact_share_ratio],how="horizontal")
combine1

xact,share,xact/share
f64,f64,f64
40.865631,56.454788,0.723865


In [41]:
ram_xact_share= (user_analysis.lazy().filter(pl.col("msg").is_in(ram_msg_list))
.group_by(["ip","us_id","msg"]).agg(pl.col("num_xact_of_users").sum().alias("xact"),pl.col("num_share_of_users").sum().alias("share"))
.group_by("msg").agg(pl.col("xact").mean(),pl.col("share").mean()).sort("xact",descending = True)
).collect()
ram_xact_share

msg,xact,share
str,f64,f64
"""00023""",92.521947,83.708245
"""00033""",89.172332,67.710625
"""10023""",82.536562,70.735986
"""00123""",81.945288,66.255817
"""10033""",78.992428,55.918678
…,…,…
"""10123""",72.153702,54.958918
"""20033""",70.25762,47.8625
"""00233""",66.677548,40.037019


In [42]:
ram_xact_share_mean= ram_xact_share.select(pl.col("xact").mean(),pl.col("share").mean())
ram_xact_share_ratio=ram_xact_share.select([(pl.col("xact").mean()/pl.col("share").mean()).alias("xact/share")])
combine = pl.concat([ram_xact_share_mean,ram_xact_share_ratio],how="horizontal")
combine

xact,share,xact/share
f64,f64,f64
76.23261,56.499691,1.349257


In [79]:
ram_ratio_mean = ram_ratio.select(pl.col("mean_num_xact").mean(),pl.col("num_xact_of_users").mean(),pl.col("num_share_of_users").mean(),pl.col("xact_share_ratio").mean())
ram_ratio_mean

ColumnNotFoundError: mean_num_xact

Resolved plan until failure:

	---> FAILED HERE RESOLVING 'select' <---
SELECT [[(col("num_xact_of_users").mean()) / (col("num_share_of_users").mean())].alias("xact_share_ratio")]
FROM
  FILTER col("msg").is_in([["00003", "00013", … "10012"]])
  FROM
    DF ["ip", "us_id", "msg", "user_id", ...]; PROJECT */7 COLUMNS

In [ ]:
re_ram_msg = pl.read_parquet("../../experiments/twitter_81306/reverse_msg_us_an.parquet")
action_stats_per_msg = (re_ram_msg.lazy().filter(pl.col("msg").is_in(reverse_msg_list)).group_by("msg").agg(
    pl.col("num_xact_of_users").mean(),
    pl.col("num_share_of_users").mean())
)
ram =lf_rob_details.join(action_stats_per_msg.collect(),left_on="msg", right_on="msg").sort(pl.col("mean_num_xact"), descending=True)
re_ram_ratio = ram.lazy().select(pl.col("msg"),pl.col("mean_num_xact"),pl.col("num_xact_of_users"),pl.col("num_share_of_users"),(pl.col("num_xact_of_users") / pl.col("num_share_of_users")).alias("xact_share_ratio")).collect()
re_ram_ratio

msg,mean_num_xact,num_xact_of_users,num_share_of_users,xact_share_ratio
str,f64,f64,f64,f64
"""13333""",176.582812,0.200084,0.208078,0.96158
"""02233""",175.57825,0.114405,0.280044,0.408525
"""01332""",171.856437,0.101695,0.294872,0.344879
"""01233""",169.0435,0.08959,0.292721,0.306059
"""00332""",168.221312,0.078627,0.307673,0.255555
"""00233""",163.912813,0.069287,0.303262,0.228472
"""02333""",144.857687,0.12186,0.269813,0.451648
"""03333""",142.906688,0.151186,0.257636,0.586821
"""01333""",139.436688,0.094801,0.281407,0.336882


['30000',
 '31000',
 '32000',
 '30001',
 '20000',
 '22000',
 '21000',
 '33000',
 '31001',
 '30100']

In [46]:
re_ram_ratio_mean = re_ram_ratio.select(pl.col("mean_num_xact").mean(),pl.col("num_xact_of_users").mean(),pl.col("num_share_of_users").mean(),pl.col("xact_share_ratio").mean())
re_ram_ratio_mean

mean_num_xact,num_xact_of_users,num_share_of_users,xact_share_ratio
f64,f64,f64,f64
159.080519,0.109499,0.279096,0.412902


逆の順番に情報を提供すると

In [16]:
re_ram_xact_mean= user_analysis.lazy().filter(pl.col("msg") == "00033").select(pl.col("num_xact_of_users").mean())
re_ram_share_mean= user_analysis.lazy().filter(pl.col("msg") == "00033").select(pl.col("num_share_of_users").mean())

re_ram_xact_mean.collect()
re_ram_share_mean.collect()

num_share_of_users
f64
0.315331


最適戦略

In [5]:
opt_ratio = (opt_ratio_lazy.lazy().group_by(["msg","us_id","ip"])
        .agg(pl.col("num_xact_of_users").mean().alias("mean_num_xact_of_users"),
             pl.col("num_share_of_users").mean().alias("mean_num_share_of_users"),
            (pl.col("num_xact_of_users").mean() / pl.col("num_share_of_users").mean()).alias("xact_share_ratio")
        )
        .collect())
opt_ratio

msg,us_id,ip,mean_num_xact_of_users,mean_num_share_of_users,xact_share_ratio
str,u64,u64,f64,f64,f64
"""30010""",13,79973,0.134906,0.148113,0.910828
"""30000""",5,69023,0.06342,0.051303,1.23619
"""30000""",4,79973,0.047638,0.054216,0.878664
"""30002""",13,12605,0.02523,0.023032,1.095413
"""32221""",3,33649,0.45,0.71,0.633803
…,…,…,…,…,…
"""21113""",8,68985,0.226,0.247333,0.913747
"""33000""",11,38598,0.388505,0.373066,1.041383
"""30103""",9,9199,0.057205,0.092773,0.61661
